# Sentiment Analysis Using Word2Vec Embeddings and KNearest Neighbors (KNN)

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

This project extends an earlier phase of the same experimentation lab, which used frequency-based representations (Bag-of-Words and TF-IDF) to establish classical baselines. In this phase, I move to **dense, distributed word representations** using **Word2Vec**, and investigate how context-aware embeddings compare to frequency-based features for sentiment classification.

The primary objective is not only to achieve strong predictive performance, but to systematically examine how embedding hyperparameters, sentence-vector aggregation strategy, and preprocessing choices interact — and to understand *why* certain configurations outperform others, rather than just reporting which one wins.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:
* Positive
* Negative

The same train/test split used in the BoW/TF-IDF phase is reused here, with Word2Vec trained only on the training partition to avoid data leakage.

---

## Why Word2Vec, and Why the Preprocessing Changes

Frequency-based methods like BoW and TF-IDF treat words as independent, count-based features — they have no notion of word meaning or context. Word2Vec instead learns dense vector representations from the **local context** each word appears in, capturing semantic relationships (e.g., "great" and "wonderful" ending up close in vector space).

This shift in representation motivates a deliberate shift in preprocessing:

* **Stopwords are retained** (rather than removed, as in the TF-IDF phase), since function words supply the local context Word2Vec learns from — removing them shrinks the effective context window and can degrade embedding quality.
* **No lemmatization** is applied, to preserve distinctions between word forms (e.g., "acting" vs "act") that may carry different contextual signal.
* **Negation handling** carries over from the earlier phase's preprocessing discipline, since negation is critical to sentiment polarity.

---

## Project Pipeline

### 1. Text Preprocessing
* Contraction expansion
* Lowercasing
* Possessive stripping and regex-based cleaning
* Stopwords retained (no removal), no lemmatization
* Corpus construction as tokenized word lists

### 2. Embedding Training
* Word2Vec (Gensim) trained on the training corpus only
* Skip-gram vs. CBOW comparison
* Hyperparameter tuning: `vector_size`, `window`, `epochs`, `min_count`
* Qualitative validation via nearest-neighbor inspection (e.g., words most similar to "great")

### 3. Feature Construction (Sentence Vectors)
* Unweighted mean-pooling of word vectors per review
* TF-IDF-weighted mean-pooling, as a comparison strategy
* Out-of-vocabulary and empty-token edge-case handling

### 4. Model Training
Classical classifiers suited to dense, continuous-valued features are evaluated, including:
* Logistic Regression
* Support Vector Machines (SVM)
* Random Forest
* XGBoost
* Artificial Neural Network (ANN)
* K-Nearest Neighbors
* Gaussian Naive Bayes

(Multinomial and Bernoulli Naive Bayes are excluded in this phase, as they assume non-negative, count-like input — an assumption Word2Vec's dense, signed vectors violate.)

### 5. Model Evaluation
Performance is assessed using the same metric set as the earlier phase, for direct comparability:
* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score (computed from predicted probabilities, not hard labels)
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single embedding and calling it final, this notebook follows the same experimentation-driven methodology as the BoW/TF-IDF phase. Word2Vec's hyperparameters were tuned incrementally — one dimension at a time — across:

* `sg` (skip-gram vs. CBOW)
* `window` size
* `vector_size`
* `epochs`

---

## Key Learning Objectives

Through this phase, I aim to:
* Understand how context-based embeddings differ from frequency-based representations, both conceptually and in preprocessing requirements.
* Empirically evaluate how embedding hyperparameters (context window, training epochs, training algorithm) affect downstream classification quality.
* Test the assumption that frequency-based reweighting (TF-IDF) improves embedding-based sentence vectors, and report the result honestly whether or not it confirms the initial hypothesis.
* Compare Word2Vec-based classical models against the BoW/TF-IDF baselines established earlier in the project.
* Build toward sequence-aware models (BiLSTM) that consume these same embeddings, as the next stage beyond static sentence-vector pooling.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)

# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [3]:
dataset = pd.read_csv('/kaggle/input/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis/MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
import re
import contractions

corpus = []
for i in range(0, len(dataset)):
    review = dataset['review'].iloc[i]
    # Fix contractions (don't -> do not)
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    # remove possessive 's before stripping other punctuation
    review = re.sub(r"'s\b", '', review)
    # Keep only letters and spaces
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split into tokens
    words = review.split()
    # NO stopword removal (keep context), NO lemmatization (keep word forms distinct)
    tokens = words
    corpus.append(tokens)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the Word2Vec model**

In [14]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=10,
    min_count=2,
    sg=1,          # skip-gram, per your established methodology
    epochs=20,
    seed=0,
    workers=1      # set to 1 for reproducibility; multi-worker training is non-deterministic even with a seed
)

## **Sentiment Similarity Test:** Iterates through sample sentiment words, checks for vocabulary presence, and prints the top 5 most similar words based on cosine distance using `model.wv.most_similar()`.

In [15]:
test_words = [
    'excellent',
    'wonderful',
    'fantastic',
    'terrible',
    'awful',
    'boring',
    'amazing'
]

for word in test_words:
    if word in model.wv:
        print(f"\nSimilar words to '{word}':")
        for similar_word, score in model.wv.most_similar(word, topn=5):
            print(f"  {similar_word}: {score:.4f}")


Similar words to 'excellent':
  great: 0.7920
  outstanding: 0.7856
  brilliant: 0.7790
  fantastic: 0.7734
  superb: 0.7709

Similar words to 'wonderful':
  great: 0.8195
  fantastic: 0.7420
  excellent: 0.7379
  perfect: 0.7294
  brilliant: 0.7116

Similar words to 'fantastic':
  amazing: 0.7824
  great: 0.7805
  excellent: 0.7734
  wonderful: 0.7420
  brilliant: 0.7293

Similar words to 'terrible':
  awful: 0.8661
  horrible: 0.8549
  bad: 0.7994
  atrocious: 0.7870
  apalling: 0.7559

Similar words to 'awful':
  terrible: 0.8661
  horrible: 0.8288
  bad: 0.7849
  atrocious: 0.7584
  apalling: 0.7371

Similar words to 'boring':
  dull: 0.8151
  pointless: 0.7959
  predictable: 0.7382
  tedious: 0.7361
  unsubstantial: 0.7277

Similar words to 'amazing':
  incredible: 0.7846
  fantastic: 0.7824
  awesome: 0.7433
  great: 0.7370
  excellent: 0.7219


## Mean Word Vector Aggregation (Dense Feature Extraction)

Translates variable-length token sequences into fixed-dimensional document embeddings for traditional machine learning models.

* **Vocabulary Lookup:** Filters input tokens against `model.wv` to retrieve corresponding Word2Vec embeddings.
* **Centroid Calculation:** Computes the element-wise mean (`axis=0`) over valid word vectors, producing a unigram centroid representation per text sample.
* **Zero-Vector Fallback:** Returns a 300-dimensional zero vector for samples containing exclusively out-of-vocabulary (OOV) terms.
* **Feature Matrix Generation:** Transforms train and test splits into 2D NumPy arrays of shape `(n_samples, 300)`.

In [ ]:
def get_average_vector(tokens, model, vector_size):
    # Only keep words that are actually in the trained vocabulary
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(valid_vectors) == 0:
        # Edge case: review has no words in vocab (rare, but handle it)
        return np.zeros(vector_size)
    
    return np.mean(valid_vectors, axis=0)

vector_size = 200
X_train_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_train])
X_test_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_test])

print("X_train shape:", X_train_w2v.shape)

## TF-IDF Weighted Document Embeddings (No Data Leakage)

Combines semantic word embeddings with statistical term frequency weights to generate rich document-level feature representations.

* **Leakage-Free Fitting:** Fits `TfidfVectorizer` exclusively on `X_train` strings, extracting Inverse Document Frequency (IDF) weights into `tfidf_weights`.
* **Fallback Strategy:** Maps missing or single-character tokens (filtered by scikit-learn's default token regex) to `1.0`, matching the theoretical minimum IDF baseline ($\ln(1) + 1$).
* **Weighted Aggregation:** Computes `np.average(vectors, axis=0, weights=weights)` to prioritize high-information, domain-specific terms while down-weighting ubiquitous words.
* **Output Matrices:** Transforms train and test token lists into dense 2D feature matrices (`X_train_tfidf_w2v` and `X_test_tfidf_w2v`) of shape `(n_samples, vector_size)`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on raw string texts (train only, no leakage)
tfidf = TfidfVectorizer()
tfidf.fit([' '.join(tokens) for tokens in X_train])
tfidf_weights = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_tfidf_w2v_vector(tokens, model, vector_size):
    valid_tokens = [w for w in tokens if w in model.wv]
    if not valid_tokens:
        return np.zeros(vector_size)
    
    # sklearn's idf = ln((1+n)/(1+df)) + 1, whose theoretical minimum is exactly 1.0
    # (a word appearing in every document). TfidfVectorizer's default token_pattern
    # also drops single-char tokens (e.g. 'a', 'i'), so they won't be in tfidf_weights
    # even though they're valid Word2Vec tokens -> fall back to that minimum weight.
    weights = [tfidf_weights.get(w, 1.0) for w in valid_tokens]
    vectors = [model.wv[w] for w in valid_tokens]
    
    return np.average(vectors, axis=0, weights=weights)

vector_size = 100
X_train_tfidf_w2v = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_train])
X_test_tfidf_w2v  = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_test])

print("X_train shape:", X_train_tfidf_w2v.shape)

X_train shape: (19923, 100)


# **Feature Scaling Using StandardScaler**

Note: I explained Feature Scaling earlier.

If you want to understand when and why to use feature scaling, you can check it here: [Go to  When is Feature Scaling Necessary in Regression Models? Section](https://github.com/Hazem1695/ml-concept-briefs)

If you want to learn how to choose between different feature scaling techniques, such as StandardScaler and RobustScaler, check out my PDF: [Go to StandardScaler vs. RobustScaler Section](https://github.com/Hazem1695/ml-concept-briefs).

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_tfidf_w2v = scaler.fit_transform(X_train_tfidf_w2v)
X_test_tfidf_w2v = scaler.transform(X_test_tfidf_w2v)

# **Building Logistic Regression Classification Model**

## Training the Building Logistic Regression Classification model on the Training set

In [18]:
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors=15, metric='cosine', weights='distance')
classifier.fit(X_train_tfidf_w2v, y_train)

KNeighborsClassifier(metric='cosine', n_neighbors=15, weights='distance')

# **Predicting the Test set results**

In [19]:
y_pred = classifier.predict(X_test_tfidf_w2v)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [1 0]]


In [20]:
y_proba = classifier.predict_proba(X_test_tfidf_w2v)[:, 1]
print(np.concatenate((y_proba.reshape(len(y_proba),1), y_test.reshape(len(y_test),1)),1))

[[0.8769559  1.        ]
 [1.         1.        ]
 [0.18953139 0.        ]
 ...
 [0.06789677 0.        ]
 [0.30413218 0.        ]
 [0.66815708 0.        ]]


# **Evaluating the Model Performance**

In [21]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_proba))


accuracies = cross_val_score(estimator=classifier, X=X_train_tfidf_w2v, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[1956  564]
 [ 300 2161]]

Accuracy Score:
0.8265408552499498

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.78      0.82      2520
           1       0.79      0.88      0.83      2461

    accuracy                           0.83      4981
   macro avg       0.83      0.83      0.83      4981
weighted avg       0.83      0.83      0.83      4981


ROC-AUC Score:
0.9110835381152325

Mean Accuracy:
0.82693369472469

Standard Deviation:
0.0027665539887608494


# Word2Vec Performance Analysis for Text Classification (K-Nearest Neighbors)

# 1. Objective

The objective of this experiment was to evaluate **Word2Vec embeddings** as a feature representation for **K-Nearest Neighbors**, a purely distance-based classifier — making it a useful contrast to the earlier Word2Vec + Logistic Regression experiment, since KNN has no learned decision boundary and depends entirely on the geometry of the embedding space.

The experiment proceeded in four phases, each isolating a different variable: feature scaling and distance metric (Phase 1), skip-gram vs. CBOW (Phase 2), embedding dimensionality (Phase 3), and pooling strategy plus `k` (Phase 4).

The experiment aims to answer the following research questions:

* Does feature scaling matter for KNN on Word2Vec vectors, and does the distance metric matter as much as it did for BoW/TF-IDF KNN?
* Does skip-gram's advantage over CBOW (found in the Logistic Regression experiment) hold for a completely different classifier?
* Does increasing embedding dimensionality help KNN the way it helped Logistic Regression?
* Does TF-IDF-weighted pooling help or hurt KNN, compared to its negative effect on Logistic Regression?
* How does the best Word2Vec KNN result compare to KNN's BoW and TF-IDF results?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Phases

| Phase | What Changed | Base Embedding |
| ----- | ------------- | ---------------- |
| 1 | Scaling, distance metric, distance weighting | 100d, window=5, CBOW (`sg=0`), epochs=5 |
| 2 | Skip-gram instead of CBOW (repeats Phase 1's last two configs) | 100d, window=5, skip-gram (`sg=1`), epochs=5 |
| 3 | Vector size 100→200 (repeats the same two configs again) | 200d, window=5, skip-gram, epochs=5 |
| 4 | TF-IDF-weighted pooling instead of mean pooling, then a `k` sweep, then combined embedding+k changes | 100d, skip-gram, various window/epochs |

Every phase includes a qualitative check — nearest neighbors of "excellent" and "wonderful" — confirming the embedding space captures reasonable synonymy throughout. All embeddings use `min_count=2, seed=0, workers=1` for reproducibility, consistent with the earlier Word2Vec + Logistic Regression experiment.

No automated hyperparameter search was used — every configuration was manually specified and evaluated exactly as configured, so there's no search-vs-evaluated mismatch to check here.

---

# 3. Experimental Results

| # | Phase | Configuration | Accuracy | ROC-AUC | CV Mean | CV Std |
| - | ----- | -------------- | -------- | ------- | ------- | ------ |
| 1 | 1 | CBOW, no scaling, k=6, minkowski | 73.54% | 0.8140 | 71.75% | 0.0025 |
| 2 | 1 | CBOW, scaled, k=6, minkowski | 74.44% | 0.8180 | 72.60% | 0.0045 |
| 3 | 1 | CBOW, scaled, k=6, cosine | 76.03% | 0.8281 | 73.82% | 0.0070 |
| 4 | 1 | CBOW, scaled, k=6, cosine, distance-weighted | 75.61% | 0.8312 | 73.81% | 0.0025 |
| 5 | 2 | **Skip-gram**, scaled, k=6, cosine | 80.08% | 0.8746 | 79.28% | 0.0022 |
| 6 | 2 | Skip-gram, scaled, k=6, cosine, distance-weighted | 79.52% | 0.8770 | 79.26% | 0.0037 |
| 7 | 3 | Skip-gram, **200d**, scaled, k=6, cosine | 79.78% | 0.8709 | 79.05% | 0.0020 |
| 8 | 3 | Skip-gram, 200d, scaled, k=6, cosine, distance-weighted | 79.56% | 0.8743 | 78.92% | **0.0006** |
| 9 | 4 | Skip-gram, 100d, scaled, k=6, cosine, **TF-IDF pool** | 80.55% | 0.8762 | 79.99% | 0.0017 |
| 10 | 4 | Skip-gram, 100d, scaled, k=6, cosine, distance-weighted, TF-IDF pool | 80.65% | 0.8795 | 80.21% | 0.0027 |
| 11 | 4 | Same as #10, k=9 | 80.97% | 0.8909 | 80.38% | 0.0017 |
| 12 | 4 | Same as #10, k=11 | **81.37%** | 0.8925 | 80.58% | 0.0023 |
| 13 | 4 | Same as #10, k=13 | 81.23% | 0.8948 | **80.88%** | 0.0016 |
| 14 | 4 | k=13, window=10, epochs=10 | 82.57% | 0.9108 | 82.47% | 0.0014 |
| **15** | 4 | **k=15, window=10, epochs=20** | **82.65%** | **0.9111** | 82.69% | 0.0028 |

---

# 4. Performance Analysis

## Effect of Feature Scaling (Runs 1 vs. 2, Isolated)

| | Accuracy |
| - | -------- |
| No scaling | 73.54% |
| Scaled (StandardScaler) | 74.44% |

Scaling gave a modest but clean +0.90 point gain — expected, since KNN's distance calculations are sensitive to the relative variance of each dimension.

## Effect of Distance Metric (Runs 2 vs. 3, Isolated)

| | Accuracy |
| - | -------- |
| Minkowski (default) | 74.44% |
| Cosine | 76.03% |

A +1.59 point gain from switching to cosine — smaller than the ~10-point swing this same switch produced for BoW KNN, but still the second-largest lever in Phase 1.

## Effect of Distance Weighting (Runs 3→4 and 5→6, Two Isolated Pairs)

| Base Config | Uniform Weights | Distance Weights | Accuracy Change | ROC-AUC Change |
| ------------ | ------------------ | -------------------- | ------------------ | ------------------ |
| CBOW | 76.03% | 75.61% | -0.42 | +0.0031 |
| Skip-gram | 80.08% | 79.52% | -0.56 | +0.0024 |

A consistent small pattern: distance weighting slightly *hurts* accuracy but slightly *improves* ROC-AUC, in both isolated tests.

## Effect of Skip-gram vs. CBOW (Runs 3 vs. 5, and 4 vs. 6, Two Isolated Pairs)

| Base Config | CBOW | Skip-gram | Change |
| ------------ | ----- | ---------- | ------ |
| Cosine, uniform | 76.03% | 80.08% | **+4.05** |
| Cosine, distance-weighted | 75.61% | 79.52% | **+3.91** |

An even larger skip-gram advantage than the Logistic Regression experiment found (+2.77 points there). This is now confirmed across two different classifiers with completely different decision mechanisms — strong, consistent evidence that `sg=1` matters regardless of what's downstream of the embedding.

## Effect of Vector Size — Diverges from the Logistic Regression Result (Runs 5 vs. 7, and 6 vs. 8)

| Base Config | 100d | 200d | Change |
| ------------ | ----- | ----- | ------ |
| Cosine, uniform | 80.08% | 79.78% | **-0.30** |
| Cosine, distance-weighted | 79.52% | 79.56% | +0.04 (negligible) |

**This directly contradicts the Logistic Regression Word2Vec experiment**, where increasing vector size from 100→300 produced a small but consistent *gain*. For KNN, doubling the dimensionality was flat-to-negative. This is a plausible curse-of-dimensionality effect: as dimensionality grows, distances between points in the embedding space become less discriminative (a well-known property of nearest-neighbor methods), while a linear classifier like Logistic Regression can learn to simply ignore or down-weight less-useful dimensions via its coefficients. KNN has no equivalent mechanism — every dimension contributes equally to the distance calculation regardless of how informative it is.

## Effect of Pooling Method — Also Diverges from the Logistic Regression Result (Runs 5 vs. 9, and 6 vs. 10)

| Base Config | Mean Pooling | TF-IDF-Weighted Pooling | Change |
| ------------ | -------------- | -------------------------- | ------ |
| Cosine, uniform | 80.08% | 80.55% | **+0.47** |
| Cosine, distance-weighted | 79.52% | 80.65% | **+1.13** |

**This is the opposite direction from the Logistic Regression experiment**, where TF-IDF-weighted pooling consistently *hurt* performance (-0.48 to -0.93 points on the same technique). For KNN, it *helped*, and by a larger margin than it hurt Logistic Regression. A plausible explanation: TF-IDF weighting sharpens which words dominate a document's pooled vector by down-weighting common terms — for a distance/similarity-based method like KNN, this directly makes cosine similarity between documents more discriminative (two reviews about the same rare topic now land closer together). A linear classifier doesn't benefit the same way, since it's already learning its own per-dimension weights during training — TF-IDF's reweighting and the classifier's own learned weights may partially fight each other for Logistic Regression, whereas KNN has no learned weights to conflict with.

**This divergence is arguably the single most useful finding across both Word2Vec experiments**: neither "TF-IDF pooling helps" nor "TF-IDF pooling hurts" is a general truth — it depends entirely on whether the downstream classifier is distance-based or coefficient-based.

## Effect of k (Runs 9-13, Same Embedding and Pooling)

| k | Accuracy | CV Mean |
| - | -------- | ------- |
| 6 | 80.65% | 80.21% |
| 9 | 80.97% | 80.38% |
| **11** | **81.37%** | 80.58% |
| 13 | 81.23% | **80.88%** |

Test accuracy peaks at k=11, but CV mean keeps climbing through k=13 — the same kind of test-accuracy-vs-CV-mean disagreement seen in a few other reports across this project. The CV mean is generally the more trustworthy signal, so k=13 (or higher, untested) may be the better choice for a production model even though k=11 wins on this specific test split.

## Combined Embedding + k Changes (Runs 13→14→15 — Not Isolated)

The final two runs change the Word2Vec configuration (window, epochs) *and* `k` simultaneously, so their gains can't be cleanly attributed to either variable alone:

| Run | Change | Accuracy |
| --- | ------- | -------- |
| 13 | Baseline (window=5, epochs=5, k=13) | 81.23% |
| 14 | + window=10, epochs=10 (k stays 13) | 82.57% (+1.34) |
| 15 | + epochs=20, k=15 | 82.65% (+0.08) |

Run 13→14 is at least a partially clean comparison (only the embedding changed, k held at 13) and shows a solid gain — consistent with the Logistic Regression experiment's finding that both wider windows and more epochs help. Run 14→15 changes both epochs and k together, so the small final gain can't be attributed to either alone.

---

# 5. Precision and Recall Analysis

### Run 5 (Skip-gram, Mean Pooling, k=6)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.80 | 0.80 | 0.80 |
| 1 | 0.80 | 0.80 | 0.80 |

Perfectly balanced — a clean baseline once skip-gram and cosine distance are in place.

### Run 9 (Skip-gram, TF-IDF Pooling, k=6)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.81 | 0.80 | 0.81 |
| 1 | 0.80 | 0.81 | 0.80 |

### Best Model — Run 15

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.87 | 0.78 | 0.82 |
| 1 | 0.79 | 0.88 | 0.83 |

The best model shows a more pronounced precision/recall asymmetry (9-point gaps) than the mid-experiment runs — the same kind of trade-off seen in several of the strongest tree-based models earlier in the project, where the top-accuracy configuration isn't always the most class-balanced one.

---

# 6. Cross-Validation Analysis

| Run | Configuration | CV Mean | CV Std |
| --- | -------------- | ------- | ------ |
| 8 | 200d, distance-weighted | 78.92% | **0.0006** |
| 13 | k=13, TF-IDF pool | 80.88% | 0.0016 |
| **15** | **Best model** | **82.69%** | 0.0028 |

Run 8 has the tightest CV std of the entire experiment, though it's far from the best accuracy — consistent with the general pattern across this project that the most stable configuration and the highest-scoring configuration are rarely the same one.

---

# 7. Best Model Configuration

```python
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

model = Word2Vec(
    sentences=X_train, vector_size=100, window=10,
    min_count=2, sg=1, epochs=20, seed=0, workers=1,
)

tfidf = TfidfVectorizer()
tfidf.fit([' '.join(tokens) for tokens in X_train])
tfidf_weights = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_tfidf_w2v_vector(tokens, model, vector_size):
    valid_tokens = [w for w in tokens if w in model.wv]
    if not valid_tokens:
        return np.zeros(vector_size)
    weights = [tfidf_weights.get(w, 1.0) for w in valid_tokens]
    vectors = [model.wv[w] for w in valid_tokens]
    return np.average(vectors, axis=0, weights=weights)

X_train_vec = np.array([get_tfidf_w2v_vector(t, model, 100) for t in X_train])
X_test_vec = np.array([get_tfidf_w2v_vector(t, model, 100) for t in X_test])

scaler = StandardScaler()
X_train_vec = scaler.fit_transform(X_train_vec)
X_test_vec = scaler.transform(X_test_vec)

classifier = KNeighborsClassifier(n_neighbors=15, metric='cosine', weights='distance')
classifier.fit(X_train_vec, y_train)
```

Performance:

* Accuracy = **82.65%**
* Precision (Class 0 / 1) = **0.87 / 0.79**
* Recall (Class 0 / 1) = **0.78 / 0.88**
* ROC-AUC = **0.9111**
* Cross-Validation Accuracy = **82.69%**
* Cross-Validation Standard Deviation = **0.0028**

---

# 8. Comparison with KNN on BoW and TF-IDF

| Representation | Best Config | Accuracy | ROC-AUC |
| ---------------- | ------------ | -------- | ------- |
| BoW | 30K features, k=15, cosine, distance | 76.79% | 0.7671 |
| TF-IDF | 30K features, k=19, cosine, distance | 80.41% | 0.8047 |
| **Word2Vec** | **100d, window=10, epochs=20, k=15, cosine, distance, TF-IDF pool** | **82.65%** | **0.9111** |

Word2Vec is the clear winner on both accuracy and ROC-AUC for KNN — a clean, monotonic improvement from BoW through TF-IDF to Word2Vec. This makes sense given Section 4's finding that KNN is fundamentally a geometry-dependent method: a representation where semantically similar words sit close together (which Word2Vec provides and sparse count-based representations don't) should help a distance-based classifier more directly than it helps a classifier that learns its own feature weights.

---

# 9. Discussion

This experiment's value comes largely from what it reveals *by contrast* with the earlier Word2Vec + Logistic Regression report, since both hold the embedding family constant and vary only the downstream classifier.

One finding replicates cleanly across both classifiers: skip-gram beats CBOW by a wide margin. Two findings *reverse* between classifiers: more embedding dimensions helped Logistic Regression but not KNN (a curse-of-dimensionality effect specific to distance-based methods), and TF-IDF-weighted pooling hurt Logistic Regression but helped KNN (likely because it sharpens document geometry in a way that benefits a similarity-based method but partially conflicts with a linear classifier's own learned weights).

The practical lesson: **when choosing embedding hyperparameters or a pooling strategy, the choice should account for what kind of classifier will consume the vectors** — a "best embedding configuration" tuned against one classifier doesn't necessarily transfer to another, even holding the embedding training itself constant.

---

# 10. Final Conclusion

This experiment evaluated Word2Vec embeddings for K-Nearest Neighbors across four phases and fifteen configurations. The best model used a **100-dimensional skip-gram embedding (window=10, 20 epochs), TF-IDF-weighted pooling, feature scaling, and `KNeighborsClassifier(n_neighbors=15, metric='cosine', weights='distance')`**, achieving **82.65% accuracy** and **0.9111 ROC-AUC** — the best result for KNN across any representation tested in this project (BoW: 76.79%, TF-IDF: 80.41%).

The most valuable finding is methodological rather than a single number: comparing this experiment against the Word2Vec + Logistic Regression report shows that **embedding dimensionality and pooling strategy affect distance-based and coefficient-based classifiers in opposite directions** — a reminder that embedding configuration should be tuned per downstream model, not assumed to transfer.